# Open Network with lognormal service times

In this notebook we will create a `ciw` network model from a specification in a JSON file.

The model represents a simple three node network problem with lognormal service times.


## 1. Imports

### 1.1 `json2ciw` imports

**We will use:**

*  `load_three_node_network_model` function that loads the built in urgent care call centre JSON file.
*  `ProcessModel`: a `pydantic` schema that provides automatic validation of the JSON.
* `CiwConverter` that accepts a valid `ProcessModel` that represents a DES model and returns a `ciw` parameter `dict`.
* `multiple_replications`:runs the network model and results a `Dataframe` of replication results.
* `summarise_results`: provides a formatted table of mean results for each node in the network.
* `tidy_to_wide_format`: the results from `multiple_replications` are returned in tidy (stacked row) format. This function converts into wide format i.e. each column is a performance measure.
* `create_user_filtered_hist` generates a interactive plotly chart to view replications for each performance measure. It requires replication data in **wide** format.

In [1]:
from json2ciw.datasets import load_three_node_network_model
from json2ciw.engine import (
    CiwConverter,
    multiple_replications
)
from json2ciw.results import (
    summarise_results, 
)

from json2ciw.schema import ProcessModel

### 1.2 Other imports

In [2]:
import ciw
from IPython.display import JSON
from rich import print

## 2. Load JSON

In [3]:
json_network = load_three_node_network_model()

display as collapsible JSON.  Expand to view the details.

In [4]:
JSON(json_network)

<IPython.core.display.JSON object>

## 3. Convert to a ProcessModel

A `ProcessModel` is a `pydantic` schema. It is independent of `ciw`. This early version will automatically validate that the JSON file is correct and that all transitions add up to 1.0 when it is created.  

In [5]:
model_instance = ProcessModel(**json_network)

The contents of the process model can be manually inspected by a developer as follows:

In [6]:
print(model_instance)

ProcessModel(
    name='Three Node Network',
    description='A simple three node network to convert to ciw',
    activities=[
        Activity(
            name='Node 1',
            type='activity',
            resource=Resource(name='Resource 1', capacity=18),
            service_distribution=Distribution(type='lognormal', parameters={'mean': 1.0, 'std': 0.1}),
            arrival_distribution=Distribution(type='exponential', parameters={'rate': 1.0})
        ),
        Activity(
            name='Node 2',
            type='activity',
            resource=Resource(name='Resource 2', capacity=12),
            service_distribution=Distribution(type='normal', parameters={'mean': 1.0, 'std': 0.1}),
            arrival_distribution=Distribution(type='exponential', parameters={'rate': 4.0})
        ),
        Activity(
            name='Node 3',
            type='activity',
            resource=Resource(name='Resource 3', capacity=25),
            service_distribution=Distribution(type='lognormal', parameters={'mean': 1.0, 'std': 0.1}),
            arrival_distribution=Distribution(type='exponential', parameters={'rate': 3.0})
        )
    ],
    transitions=[
        Transition(source='Node 1', target='Exit', probability=0.1),
        Transition(source='Node 1', target='Node 2', probability=0.6),
        Transition(source='Node 1', target='Node 3', probability=0.3),
        Transition(source='Node 2', target='Exit', probability=0.6),
        Transition(source='Node 2', target='Node 1', probability=0.1),
        Transition(source='Node 2', target='Node 3', probability=0.3),
        Transition(source='Node 3', target='Exit', probability=0.2),
        Transition(source='Node 3', target='Node 2', probability=0.4),
        Transition(source='Node 3', target='Node 1', probability=0.4)
    ]
)

In [7]:
model_instance.display_diagram(include_resources=False)

```mermaid
graph TD
    %% Three Node Network: A simple three node network to convert to ciw
    Arrivals_Node_1("Time between arrivals<br/>Exponential(λ=1.0)")
    Arrivals_Node_2("Time between arrivals<br/>Exponential(λ=4.0)")
    Arrivals_Node_3("Time between arrivals<br/>Exponential(λ=3.0)")
    Node_1["Node 1<br/>Lognormal(mean=1.0, stdev=0)"]
    Node_2["Node 2<br/>Normal(mean=1.0, sd=0)"]
    Node_3["Node 3<br/>Lognormal(mean=1.0, stdev=0)"]
    Exit(["Exit"])

    Arrivals_Node_1 --> Node_1
    Arrivals_Node_2 --> Node_2
    Arrivals_Node_3 --> Node_3
    Node_1 -->|10%| Exit
    Node_1 -->|60%| Node_2
    Node_1 -->|30%| Node_3
    Node_2 -->|60%| Exit
    Node_2 -->|10%| Node_1
    Node_2 -->|30%| Node_3
    Node_3 -->|20%| Exit
    Node_3 -->|40%| Node_2
    Node_3 -->|40%| Node_1
```

In [8]:
model_instance.display_diagram(include_resources=True)

```mermaid
graph TD
    %% Three Node Network: A simple three node network to convert to ciw
    Arrivals_Node_1("Time between arrivals<br/>Exponential(λ=1.0)")
    Arrivals_Node_2("Time between arrivals<br/>Exponential(λ=4.0)")
    Arrivals_Node_3("Time between arrivals<br/>Exponential(λ=3.0)")
    Node_1["Node 1<br/>Lognormal(mean=1.0, stdev=0)"]
    Node_2["Node 2<br/>Normal(mean=1.0, sd=0)"]
    Node_3["Node 3<br/>Lognormal(mean=1.0, stdev=0)"]
    Resource_Resource_1(("Resource 1<br/>(18)"))
    Resource_Resource_2(("Resource 2<br/>(12)"))
    Resource_Resource_3(("Resource 3<br/>(25)"))
    Exit(["Exit"])

    Arrivals_Node_1 --> Node_1
    Arrivals_Node_2 --> Node_2
    Arrivals_Node_3 --> Node_3
    Resource_Resource_1 -.Seize.-> Node_1
    Node_1 -.Release.-> Resource_Resource_1
    Resource_Resource_2 -.Seize.-> Node_2
    Node_2 -.Release.-> Resource_Resource_2
    Resource_Resource_3 -.Seize.-> Node_3
    Node_3 -.Release.-> Resource_Resource_3
    Node_1 -->|10%| Exit
    Node_1 -->|60%| Node_2
    Node_1 -->|30%| Node_3
    Node_2 -->|60%| Exit
    Node_2 -->|10%| Node_1
    Node_2 -->|30%| Node_3
    Node_3 -->|20%| Exit
    Node_3 -->|40%| Node_2
    Node_3 -->|40%| Node_1
```

In [9]:
# summary of distributions
model_instance.get_distributions_df()

,Activity,Phase,Distribution Type,Parameters
0,Node 1,Arrival,Exponential,rate=1.0
1,Node 1,Service,Lognormal,"mean=1.0, std=0.1"
2,Node 2,Arrival,Exponential,rate=4.0
3,Node 2,Service,Normal,"mean=1.0, std=0.1"
4,Node 3,Arrival,Exponential,rate=3.0
5,Node 3,Service,Lognormal,"mean=1.0, std=0.1"


In [10]:
# summary of routing percentages
model_instance.get_routing_matrix_df()

,Node 1,Node 2,Node 3,Exit
Source Activity,,,,
Node 1,0.0,0.6,0.3,0.1
Node 2,0.1,0.0,0.3,0.6
Node 3,0.4,0.4,0.0,0.2


In [11]:
# resources
model_instance.get_resources_df()

,Resource,Activity,Count
0,Resource 1,Node 1,18
1,Resource 2,Node 2,12
2,Resource 3,Node 3,25


## 3. Convert to a `ciw` Network Model

In [12]:
adapter = CiwConverter(model_instance)
network_params = adapter.generate_params()

The variable `network_params` is a python `dict` that contains all the parameters that `ciw` requires for a very simple queuing model. This can be passed as keyword args to the `ciw.create_network` function.

In [13]:
network_params

{'number_of_servers': [18, 12, 25],
 'arrival_distributions': [Exponential(rate=1.0),
  Exponential(rate=4.0),
  Exponential(rate=3.0)],
 'service_distributions': [Lognormal(mean=-0.0049751654265839124, sd=0.0997513451195916),
  Normal(mean=1.0, sd=0.1),
  Lognormal(mean=-0.0049751654265839124, sd=0.0997513451195916)],
 'routing': [[0.0, 0.6, 0.3], [0.1, 0.0, 0.3], [0.4, 0.4, 0.0]]}

In [14]:
# we use ciw's native `create_network` method
network = ciw.create_network(**network_params)
print(type(network))

<class 'ciw.network.Network'>

In [15]:
# Run a quick simulation to verify it works without crashing...
sim = ciw.Simulation(network)
sim.simulate_until_max_time(50)
print("Quick simulation run worked!")

Quick simulation run worked!

## 4. Run the model for multiple replications

The `multiple_replications` function returns a `DataFrame` that contains one row per activity per replication.  So if there are two nodes there are two rows for each replication.

In [16]:
# Run replications with activity/resource names
df_reps = multiple_replications(
    network, 
    model_instance, 
    num_reps=100, 
    runtime=2880,
    warmup=1440,
    n_jobs=-1 # parallel reps
)

df_reps.head()

,rep,node_id,activity_name,resource_name,resource_capacity,arrivals,mean_wait,mean_service,utilisation,mean_Lq
0,0,1,Node 1,Resource 1,18,7216,0.000000,1.000161,28.238367,0.000000
1,0,2,Node 2,Resource 2,12,14546,0.138830,1.000103,83.861598,1.402375
2,0,3,Node 3,Resource 3,25,10838,0.000000,0.999720,30.372911,0.000000
3,1,1,Node 1,Resource 1,18,7365,0.000000,1.001236,27.770561,0.000000
4,1,2,Node 2,Resource 2,12,14515,0.182148,1.000512,83.455292,1.836026


Filter the nodes with the usual `pandas` syntax

In [17]:
df_reps.loc[df_reps['activity_name'] == "Service 1"].head(3)

,rep,node_id,activity_name,resource_name,resource_capacity,arrivals,mean_wait,mean_service,utilisation,mean_Lq


## 5. Summarise results

If needed there is a built in function to create a `DataFrame` that contains a mean summary of all metrics across the nodes.

In [18]:
# Get summary table
summary = summarise_results(df_reps)
summary.round(1)

activity,Metric,Node 1 (Resource 1),Node 2 (Resource 2),Node 3 (Resource 3)
0,Mean arrivals,7204.1,14395.1,10785.4
1,Mean waiting time,0.0,0.1,0.0
2,Mean service time,1.0,1.0,1.0
3,Mean utilisation,27.8,83.3,30.0
4,Mean queue length,0.0,1.4,0.0
